## Graphs

### Creation of age_distribution.json

General distribution showing what the mean age of athletes per sport is

In [3]:
import pandas as pd
import json
from collections import defaultdict

df = pd.read_csv("data.csv", sep=";")

df["DOB"]  = pd.to_datetime(df["DOB"],  errors="coerce")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Sex"]  = df["Sex"].str.strip().str.lower()
df["Event"] = df["Event"].str.strip()


df = df.dropna(subset=["DOB", "Date"])

def age_at(dob, date):
    age = date.year - dob.year
    if (date.month, date.day) < (dob.month, dob.day):
        age -= 1
    return age

df["age"] = df.apply(lambda r: age_at(r["DOB"], r["Date"]), axis=1)

df = df[(df["age"] >= 10) & (df["age"] <= 80)]

output = {}

for event, group in df.groupby("Event"):
    event_data = {}

    for sex in ["male", "female", "all"]:
        subset = group if sex == "all" else group[group["Sex"] == sex]
        if subset.empty:
            continue

        ages = subset["age"].tolist()
        min_age, max_age = int(min(ages)), int(max(ages))

        # 1-year bins
        bins = {age: 0 for age in range(min_age, max_age + 1)}
        for a in ages:
            bins[int(a)] += 1

        event_data[sex] = {
            "bins":   [{"age": k, "count": v} for k, v in sorted(bins.items())],
            "stats": {
                "count":  len(ages),
                "min":    min_age,
                "max":    max_age,
                "mean":   round(sum(ages) / len(ages), 1),
                "median": int(sorted(ages)[len(ages) // 2]),
            }
        }

    output[event] = event_data

with open("age_distribution.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f" {len(df)} valid performances across {len(output)} events")
print(f" Saved to age_distribution.json\n")
print(f"{'Event':<35} {'All':>5} {'Male':>6} {'Female':>8}  Median age")
print("-" * 70)
for event, data in sorted(output.items()):
    counts = {s: data[s]["stats"]["count"] for s in data}
    median = data.get("all", data.get("male", data.get("female")))["stats"]["median"]
    print(
        f"{event:<35}"
        f"{counts.get('all',   0):>5}"
        f"{counts.get('male',  0):>7}"
        f"{counts.get('female',0):>8}"
        f"  {median}"
    )


✓ 455090 valid performances across 46 events
✓ Saved to age_distribution.json

Event                                 All   Male   Female  Median age
----------------------------------------------------------------------
10 Kilometres                       1676   1054     622  25
10 Kilometres Race Walk             2263    360    1903  24
10 Miles Road                        296    137     159  26
100 Metres                         51128  24505   26623  24
100 Metres Hurdles                  7544      0    7544  26
1000 Metres                         1017    505     512  25
10000 Metres                        7861   4783    3078  25
10000 Metres Race Walk               680    493     187  25
110 Metres Hurdles                 13934  13934       0  25
15 Kilometres                        383    188     195  25
1500 Metres                        18395   9009    9386  25
20 Kilometres                        339    245      94  25
20 Kilometres Race Walk            12986   3767    9219  24


### Creation of mean_age.json

General distribution showing what the mean age of athletes per gender and per sport is
It's later on used with our two .html previews :
- the general one : mean_age_chart.html
- the women/men compairaison one: mean_age_chart w&m.html

In [ ]:
import pandas as pd
import json
 
df = pd.read_csv("data.csv", sep=";")
 
df["DOB"]   = pd.to_datetime(df["DOB"],   errors="coerce")
df["Date"]  = pd.to_datetime(df["Date"],  errors="coerce")
df["Sex"]   = df["Sex"].str.strip().str.lower()
df["Event"] = df["Event"].str.strip()
 
df = df.dropna(subset=["DOB", "Date"])
 
def age_at(dob, date):
    age = date.year - dob.year
    if (date.month, date.day) < (dob.month, dob.day):
        age -= 1
    return age
 
df["age"] = df.apply(lambda r: age_at(r["DOB"], r["Date"]), axis=1)
df = df[(df["age"] >= 10) & (df["age"] <= 80)]
 

records = []
 
for event, group in df.groupby("Event"):
    row = {"event": event}
    for sex in ["male", "female"]:
        subset = group[group["Sex"] == sex]["age"]
        if not subset.empty:
            row[sex] = round(float(subset.mean()), 1)
    
    row["all"] = round(float(group["age"].mean()), 1)
    records.append(row)
 

records.sort(key=lambda r: r["all"])

with open("mean_age.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f" {len(records)} events\n")
print(f"{'Event':<35} {'All':>5}  {'Male':>5}  {'Female':>6}")
print("-" * 60)
for r in records:
    print(
        f"{r['event']:<35}"
        f"{r.get('all',   '-'):>5}"
        f"  {r.get('male',  '-'):>5}"
        f"  {r.get('female','-'):>6}"
    )

 46 events

Event                                 All   Male  Female
------------------------------------------------------------
2000 Metres Steeplechase            22.2   25.4    21.8
30 Kilometres Race Walk             23.6   23.6       -
200 Metres                          23.9   24.1    23.8
10 Kilometres Race Walk             24.1   25.2    23.9
300 Metres                          24.1   24.0    24.8
5 Kilometres Race Walk              24.2   26.7    24.0
400 Metres                          24.3   23.8    24.7
100 Metres                          24.5   24.6    24.4
3000 Metres Steeplechase            24.6   25.3    24.2
5000 Metres                         24.6   24.4    25.0
20000 Metres Race Walk              24.8   25.0    24.8
600 Metres                          24.8   24.1    25.0
High Jump                           24.8   24.6    25.7
Heptathlon                          24.9      -    24.9
1000 Metres                         25.0   24.3    25.7
Hammer Throw                  